## 1. Reconstrucción de la tabla completa

La Fase 1 dejó la unión de las 46 hojas por número de bloque solo en el notebook del EDA, sin guardarla como archivo. Para que el preprocesamiento sea reproducible desde los Excel originales, esa unión se rehace con la función `construir_tabla_completa` de `src/preprocessing_pipeline.py`, y se controla que coincida con la del EDA.


In [9]:
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

sys.path.append("../src")
from auxiliar_functions import tabla_simple
from preprocessing_pipeline import construir_tabla_completa

import importlib
import preprocessing_pipeline

importlib.reload(preprocessing_pipeline)    # vuelve a leer el archivo de src/


<module 'preprocessing_pipeline' from 'c:\\Users\\Asus\\Desktop\\Automatizaciones\\Materias Facultad\\Tercer Año - Segundo Cuatrimestre\\Laboratorio de Datos II\\Trabajo Práctico Final LD2\\notebooks\\../src\\preprocessing_pipeline.py'>

In [2]:
tabla_completa = construir_tabla_completa(Path("../data"))   # tarda unos 6 minutos: lee los tres Excel completos

control = pd.DataFrame({"Valor": {
    "Días (filas)": len(tabla_completa),
    "Columnas": tabla_completa.shape[1],
    "Días válidos": int(tabla_completa["valido"].sum()),
    "Filas con objetivo": int(tabla_completa["objetivo"].notna().sum()),
    "Columnas de las otras hojas": int(sum(" | " in str(c) for c in tabla_completa.columns)),
}})
print("Control de la tabla completa (debe coincidir con el EDA: 1.212 días, 453 columnas, 1.187 válidos, 1.186 con objetivo, 420 de otras hojas)")
display(tabla_simple(control, precision=0))


Control de la tabla completa (debe coincidir con el EDA: 1.212 días, 453 columnas, 1.187 válidos, 1.186 con objetivo, 420 de otras hojas)


,Valor
Días (filas),1212
Columnas,453
Días válidos,1187
Filas con objetivo,1186
Columnas de las otras hojas,420


## 2. División temporal en entrenamiento, validación y prueba

Antes de tomar cualquier decisión de limpieza o de selección se divide la tabla por orden de fecha, sin mezclar las filas: el 70 % más antiguo (848 días, hasta el 28/10/2022) para entrenamiento, el 15 % siguiente (182 días) para validación y el 15 % más reciente (182 días, desde el 28/04/2023) para prueba. Mezclar las filas, como hace `train_test_split`, dejaría días futuros dentro del entrenamiento; con este corte, cada criterio se mide solo con entrenamiento y la prueba se toca una única vez, al final. El corte de entrenamiento coincide con el que se usó en el EDA, así que las correlaciones y los criterios ya calculados siguen valiendo. Como el objetivo de cada día es el consumo del siguiente, el último día de cada tramo comparte una fila con el primero del tramo que sigue, un solapamiento de efecto despreciable.


In [10]:
from preprocessing_pipeline import dividir_por_fecha

X_train, X_val, X_test = dividir_por_fecha(tabla_completa)

resumen = pd.DataFrame({
    nombre: {"Días": len(parte),
             "Desde": parte["DIA"].iloc[0].strftime("%d/%m/%Y"),
             "Hasta": parte["DIA"].iloc[-1].strftime("%d/%m/%Y"),
             "Válidos con objetivo": int((parte["valido"] & parte["objetivo"].notna()).sum())}
    for nombre, parte in {"X_train": X_train, "X_val": X_val, "X_test": X_test}.items()}).T
print("División temporal de la tabla completa (70 % / 15 % / 15 %, sin mezclar las filas)")
display(tabla_simple(resumen, precision=0))


División temporal de la tabla completa (70 % / 15 % / 15 %, sin mezclar las filas)


,Días,Desde,Hasta,Válidos con objetivo
X_train,848,01/07/2020,28/10/2022,817
X_val,182,29/10/2022,27/04/2023,172
X_test,182,28/04/2023,26/10/2023,176
